In [30]:
import pandas as pd

DATA_PATH = "/content/sample_data/bbc-text.csv"
df = pd.read_csv(DATA_PATH)

print(df.shape)
print(df.columns)
df.head()

(2225, 2)
Index(['category', 'text'], dtype='object')


,category,text
0,tech,tv future in the hands of viewers with home th...
1,business,worldcom boss left books alone former worldc...
2,sport,tigers wary of farrell gamble leicester say ...
3,sport,yeading face newcastle in fa cup premiership s...
4,entertainment,ocean s twelve raids box office ocean s twelve...


In [32]:
import string

STOP_WORDS = {
    "the","a","an","and","or","is","are","was","were","to","of","in","on","at","for","with","by","from",
    "this","that","it","its","as","be","been","being","but","not","so","if","then","than","into","about",
    "over","under","after","before","between","during","while","also","can","could","would","should","will",
    "just","more","most","some","such","no","nor","only","own","same","too","very",
    "they","them","their","there","here","he","him","his","she","her","we","us","our","you","your","i","me","my",
    "who","whom","which","what","when","where","why","how",
    "has","have","had","having","do","does","did","done","doing","said","say","says","mr","mrs","ms"
}

PUNCT_TABLE = str.maketrans({ch: " " for ch in string.punctuation})

def clean_and_tokenize(text: str):
    if not isinstance(text, str):
        return []
    text = text.lower()
    text = text.translate(PUNCT_TABLE)
    tokens = text.split()
    tokens = [t for t in tokens if t.isalpha() and len(t) > 1]
    tokens = [t for t in tokens if t not in STOP_WORDS]
    return tokens

docs_tokens = df["text"].apply(clean_and_tokenize).tolist()
labels = df["category"].tolist()
classes = sorted(set(labels))

print("Classes:", classes)
print("Example tokens:", docs_tokens[0][:30])


Classes: ['business', 'entertainment', 'politics', 'sport', 'tech']
Example tokens: ['tv', 'future', 'hands', 'viewers', 'home', 'theatre', 'systems', 'plasma', 'high', 'definition', 'tvs', 'digital', 'video', 'recorders', 'moving', 'living', 'room', 'way', 'people', 'watch', 'tv', 'radically', 'different', 'five', 'years', 'time', 'according', 'expert', 'panel', 'gathered']


In [33]:
import random

indices = list(range(len(docs_tokens)))
random.seed(42)
random.shuffle(indices)

split = int(0.8 * len(indices))
train_idx, test_idx = indices[:split], indices[split:]

X_train = [docs_tokens[i] for i in train_idx]
y_train = [labels[i] for i in train_idx]
X_test  = [docs_tokens[i] for i in test_idx]
y_test  = [labels[i] for i in test_idx]

len(X_train), len(X_test)

(1780, 445)

In [34]:
from collections import Counter

def build_vocab(docs, min_freq=2):
    """
    Builds vocab from training docs only.
    Returns: vocab(word->index), freq Counter
    """
    freq = Counter()
    for tokens in docs:
        freq.update(tokens)

    vocab_words = [w for w,c in freq.items() if c >= min_freq]
    vocab_words = sorted(vocab_words)
    vocab = {w:i for i,w in enumerate(vocab_words)}
    return vocab, freq

vocab, train_freq = build_vocab(X_train, min_freq=2)
print("Vocab size:", len(vocab))

Vocab size: 16707


In [35]:
from collections import defaultdict

def doc_to_bow_sparse(tokens, vocab):
    vec = defaultdict(int)
    for w in tokens:
        if w in vocab:
            vec[vocab[w]] += 1
    return dict(vec)

def doc_to_bow_dense(tokens, vocab):
    vec = [0] * len(vocab)
    for w in tokens:
        if w in vocab:
            vec[vocab[w]] += 1
    return vec

# demo on 1 doc
demo_sparse = doc_to_bow_sparse(X_train[0], vocab)
print("Sparse BoW (first 10 items):", list(demo_sparse.items())[:10])

Sparse BoW (first 10 items): [(2255, 10), (14170, 2), (10784, 2), (12820, 6), (4127, 1), (152, 1), (8310, 1), (9535, 1), (12750, 1), (5605, 3)]


In [36]:
import math
from collections import Counter, defaultdict

def train_multinomial_nb(docs, y, vocab, alpha=1.0):
    class_doc_count = Counter(y)
    total_docs = len(y)
    log_prior = {c: math.log(class_doc_count[c] / total_docs) for c in class_doc_count}

    term_counts = {c: defaultdict(int) for c in class_doc_count}
    total_terms = {c: 0 for c in class_doc_count}

    for tokens, label in zip(docs, y):
        for w in tokens:
            if w in vocab:
                term_counts[label][w] += 1
                total_terms[label] += 1

    log_likelihood = {c: {} for c in class_doc_count}
    V = len(vocab)

    for c in class_doc_count:
        denom = total_terms[c] + alpha * V
        for w in vocab:
            count = term_counts[c].get(w, 0)
            log_likelihood[c][w] = math.log((count + alpha) / denom)

    return log_prior, log_likelihood

def predict_multinomial_nb(docs, classes, log_prior, log_likelihood, vocab):
    preds = []
    for tokens in docs:
        scores = {}
        for c in classes:
            s = log_prior[c]
            for w in tokens:
                if w in vocab:
                    s += log_likelihood[c][w]
            scores[c] = s
        preds.append(max(scores, key=scores.get))
    return preds

log_prior, log_likelihood = train_multinomial_nb(X_train, y_train, vocab, alpha=1.0)
pred_bow = predict_multinomial_nb(X_test, classes, log_prior, log_likelihood, vocab)

acc_bow = sum(p==t for p,t in zip(pred_bow, y_test)) / len(y_test)
print("BoW + NB accuracy:", acc_bow)


BoW + NB accuracy: 0.9865168539325843


In [37]:
from collections import Counter

def compute_document_frequencies(docs):
    df_counts = Counter()
    for tokens in docs:
        df_counts.update(set(tokens))
    return df_counts

def compute_idf(df_counts, N):
    idf = {}
    for term, dfc in df_counts.items():
        idf[term] = math.log((N + 1) / (dfc + 1)) + 1.0
    return idf

def doc_to_tfidf(tokens, idf):
    counts = Counter(tokens)
    total = sum(counts.values()) or 1
    vec = {}
    for term, c in counts.items():
        if term in idf:
            tf = c / total
            vec[term] = tf * idf[term]
    return vec

df_counts = compute_document_frequencies(X_train)
idf = compute_idf(df_counts, N=len(X_train))

X_train_tfidf = [doc_to_tfidf(toks, idf) for toks in X_train]
X_test_tfidf  = [doc_to_tfidf(toks, idf) for toks in X_test]

print("Example TF-IDF keys:", list(X_train_tfidf[0].items())[:10])


Example TF-IDF keys: [('carry', 0.25892640679215656), ('star', 0.03761940190919449), ('patsy', 0.08705902908077895), ('rowlands', 0.26117708724233685), ('dies', 0.032658508121086054), ('actress', 0.024717805992979007), ('known', 0.019190359084023043), ('millions', 0.023062689701676367), ('roles', 0.028590136610632335), ('films', 0.061829157614965144)]


In [38]:
from collections import defaultdict, Counter
import math

def dot_sparse(a, b):
    if len(a) > len(b):
        a, b = b, a
    return sum(v * b.get(k, 0.0) for k, v in a.items())

def norm_sparse(v):
    return math.sqrt(sum(x*x for x in v.values()))

def train_centroids(X_tfidf, y, classes):
    centroid_sum = {c: defaultdict(float) for c in classes}
    counts = Counter(y)

    for vec, label in zip(X_tfidf, y):
        for k, v in vec.items():
            centroid_sum[label][k] += v

    centroids = {}
    for c in classes:
        denom = counts[c] if counts[c] else 1
        centroids[c] = {k: v/denom for k,v in centroid_sum[c].items()}

    cent_norm = {c: norm_sparse(centroids[c]) for c in classes}
    return centroids, cent_norm

def predict_centroids(X_tfidf, classes, centroids, cent_norm):
    preds = []
    for vec in X_tfidf:
        vnorm = norm_sparse(vec) or 1e-12
        best_c, best_sim = None, -1e18
        for c in classes:
            denom = vnorm * (cent_norm[c] or 1e-12)
            sim = dot_sparse(vec, centroids[c]) / denom
            if sim > best_sim:
                best_sim = sim
                best_c = c
        preds.append(best_c)
    return preds

centroids, cent_norm = train_centroids(X_train_tfidf, y_train, classes)
pred_tfidf = predict_centroids(X_test_tfidf, classes, centroids, cent_norm)

acc_tfidf = sum(p==t for p,t in zip(pred_tfidf, y_test)) / len(y_test)
print("TF-IDF + centroid accuracy:", acc_tfidf)


TF-IDF + centroid accuracy: 0.9820224719101124


In [40]:

global_counts = Counter()
total_tokens = 0
for toks in X_train:
    global_counts.update(toks)
    total_tokens += len(toks)

tf_global = {w: c/total_tokens for w,c in global_counts.items()}

# (A) High TF, Low IDF
top_tf = sorted(tf_global.items(), key=lambda x: x[1], reverse=True)[:30]
high_tf_low_idf = sorted([(w, tf, idf.get(w, 0.0)) for w,tf in top_tf], key=lambda x: x[2])[:10]

print("High TF but Low IDF (common words):")
for w, tfv, idfv in high_tf_low_idf:
    print(f"{w:15} TF={tfv:.6f}  IDF={idfv:.3f}")

# (B) High IDF, Low TF (rare words)
candidates = [(w, tf_global[w], idf[w], global_counts[w]) for w in idf if global_counts[w] <= 2]
high_idf_low_tf = sorted(candidates, key=lambda x: x[2], reverse=True)[:10]

print("\nHigh IDF but Low TF (rare words):")
for w, tfv, idfv, cnt in high_idf_low_tf:
    print(f"{w:15} count={cnt}  TF={tfv:.8f}  IDF={idfv:.3f}")


High TF but Low IDF (common words):
up              TF=0.004680  IDF=1.584
year            TF=0.004731  IDF=1.631
one             TF=0.003891  IDF=1.727
out             TF=0.003481  IDF=1.785
new             TF=0.004061  IDF=1.835
all             TF=0.003298  IDF=1.868
last            TF=0.002865  IDF=1.896
two             TF=0.002658  IDF=1.931
time            TF=0.002736  IDF=1.990
first           TF=0.002857  IDF=2.003

High IDF but Low TF (rare words):
patsy           count=2  TF=0.00000516  IDF=7.792
bless           count=1  TF=0.00000258  IDF=7.792
beresford       count=1  TF=0.00000258  IDF=7.792
guildhall       count=1  TF=0.00000258  IDF=7.792
hove            count=1  TF=0.00000258  IDF=7.792
regulars        count=1  TF=0.00000258  IDF=7.792
sid             count=1  TF=0.00000258  IDF=7.792
carved          count=1  TF=0.00000258  IDF=7.792
cazalets        count=1  TF=0.00000258  IDF=7.792
sanzar          count=1  TF=0.00000258  IDF=7.792
